# Modul 2 — Indexing: Incidence Matrix, Inverted Index dan Positional Index

Tujuan asli modul dipertahankan. Semua contoh disesuaikan dengan **50 dokumen D01–D50**, 25 query, serta qrels skor 1–3. Qrels diperiksa validitas referensi, tetapi **tidak diperlukan untuk membangun indeks**.

**Prasyarat:** jalankan dari folder `notebooks/`; pustaka `pandas` dan `Sastrawi` terpasang. `stki_utils.py` telah disesuaikan membaca folder `../Data/` dan kolom `id_doc`.


In [1]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [2]:
from collections import Counter, defaultdict
from stki_utils import corpus_texts, preprocess, load_queries, load_qrels

teks = corpus_texts()  # {id_doc: judul + isi}
docs = {d: preprocess(t) for d, t in teks.items()}
queries = load_queries()
qrels = load_qrels()
assert len(docs) == 50 and len(queries) == 25
assert all(d in docs for labels in qrels.values() for d in labels)
print("Jumlah dokumen:", len(docs), "| queries:", len(queries), "| qrels:", sum(map(len,qrels.values())))
print("D02:", docs["D02"][:35])


Jumlah dokumen: 50 | queries: 25 | qrels: 59
D02: ['peran', 'ai', 'generatif', 'industri', 'kreatif', 'maju', 'teknologi', 'cerdas', 'buat', 'ai', 'terus', 'kembang', 'pesat', 'pengaruh', 'bagai', 'sektor', 'industri', 'masuk', 'industri', 'kreatif', 'ai', 'generatif', 'guna', 'algoritma', 'cipta', 'konten', 'baru', 'dasar', 'data', 'ubah', 'cara', 'manusia', 'interaksi', 'seni', 'musik']


## Langkah 1 — Term-document incidence matrix

Matriks biner berukuran |V| × N; nilainya 1 bila term muncul di dokumen. Enam contoh sekarang dipilih dari korpus aktual: AI (D02, D23), machine learning (D05), sistem operasi (D10), virtualisasi (D14), dan smart home (D24).


In [3]:
subset = ["D02", "D05", "D10", "D14", "D23", "D24"]
vocab_subset = sorted({t for d in subset for t in docs[d]})
incidence = pd.DataFrame([[int(t in docs[d]) for d in subset] for t in vocab_subset],
                         index=vocab_subset, columns=subset)
print("Ukuran matriks subset:", incidence.shape)
term_pilih = [t for t in ["cerdas", "buatan", "data", "sistem", "teknologi", "rumah", "virtual", "mesin"] if t in incidence.index]
display(incidence.loc[term_pilih])


Ukuran matriks subset: (167, 6)


,D02,D05,D10,D14,D23,D24
cerdas,1,0,0,0,1,0
data,1,1,0,0,1,0
sistem,0,0,1,1,0,1
teknologi,1,0,0,0,1,0
rumah,0,0,0,0,0,1
virtual,0,0,0,1,0,0
mesin,0,0,0,1,1,0


In [4]:
vocab = sorted({t for tokens in docs.values() for t in tokens})
sel_total = len(vocab) * len(docs)
sel_isi = sum(len(set(tokens)) for tokens in docs.values())
print(f"Kosakata: {len(vocab)} term, {len(docs)} dokumen")
print(f"Sel matriks penuh : {sel_total}")
print(f"Sel bernilai 1    : {sel_isi} ({sel_isi / sel_total:.1%})")

Kosakata: 1285 term, 50 dokumen
Sel matriks penuh : 64250
Sel bernilai 1    : 2314 (3.6%)


Sebagian besar sel bernilai 0 (*sparse*). Pada koleksi nyata berisi jutaan dokumen, matriks penuh tidak mungkin disimpan. Solusinya adalah **inverted index**: untuk setiap term hanya disimpan daftar dokumen yang memuatnya.

## Langkah 2 — Membangun inverted index
Struktur yang dibangun: `index[term] = {doc_id: tf}`. Panjang posting list sama dengan *document frequency* (df).

In [5]:
def build_inverted_index(docs_tokens):
    index = defaultdict(dict)
    for doc_id, tokens in docs_tokens.items():
        for term, tf in Counter(tokens).items():
            index[term][doc_id] = tf
    return dict(sorted(index.items()))

index = build_inverted_index(docs)
print("Jumlah term di kamus:", len(index))
for term in ["cerdas", "data", "sistem", "energi", "tani"]:
    postings = index.get(term, {})
    print(f"{term:8s} df={len(postings):2d}  postings={postings}")


Jumlah term di kamus: 1285
cerdas   df= 2  postings={'D02': 1, 'D23': 1}
data     df=11  postings={'D01': 1, 'D02': 1, 'D05': 1, 'D06': 2, 'D23': 3, 'D28': 2, 'D30': 1, 'D31': 3, 'D32': 6, 'D33': 1, 'D35': 1}
sistem   df=12  postings={'D09': 1, 'D10': 3, 'D14': 1, 'D16': 1, 'D17': 1, 'D19': 1, 'D24': 1, 'D25': 1, 'D26': 1, 'D32': 1, 'D33': 1, 'D34': 2}
energi   df= 4  postings={'D18': 3, 'D21': 1, 'D36': 5, 'D49': 7}
tani     df= 2  postings={'D01': 3, 'D20': 1}


In [6]:
tabel = pd.DataFrame({
    "df": {t: len(p) for t, p in index.items()},
    "cf (total tf)": {t: sum(p.values()) for t, p in index.items()},
}).sort_values("df", ascending=False)
tabel.head(10)

,df,cf (total tf)
guna,20,26
jadi,13,30
sistem,12,15
buat,12,13
data,11,22
rupa,11,15
hasil,10,17
indonesia,9,26
banyak,9,16
satu,9,11


## Langkah 3 — Menyimpan dan memuat index

In [7]:
import json, os

os.makedirs("output", exist_ok=True)
with open("output/inverted_index.json", "w") as f:
    json.dump(index, f)

with open("output/inverted_index.json") as f:
    index_dimuat = json.load(f)
print("Index tersimpan dan dimuat ulang dengan benar:", index_dimuat == index)
print("Ukuran file:", os.path.getsize("output/inverted_index.json"), "byte")

Index tersimpan dan dimuat ulang dengan benar: True
Ukuran file: 38418 byte


## Langkah 4 — Positional index dan phrase query
*Positional index* menyimpan posisi kemunculan term: `index[term][doc_id] = [posisi, ...]`. Dengan posisi, kita bisa mencari frasa, yaitu term yang muncul berurutan.

In [8]:
def build_positional_index(docs_tokens):
    idx = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in docs_tokens.items():
        for pos, term in enumerate(tokens):
            idx[term][doc_id].append(pos)
    return {t: dict(p) for t, p in sorted(idx.items())}

pos_index = build_positional_index(docs)
print("cerdas:", pos_index.get("cerdas", {}))
print("data:", pos_index.get("data", {}))


cerdas: {'D02': [7], 'D23': [13]}
data: {'D01': [21], 'D02': [28], 'D05': [9], 'D06': [67, 80], 'D23': [26, 56, 77], 'D28': [7, 17], 'D30': [11], 'D31': [0, 2, 9], 'D32': [0, 3, 11, 23, 31, 41], 'D33': [17], 'D35': [24]}


In [9]:
def phrase_search(frasa, pos_index, tokenizer=preprocess):
    terms = tokenizer(frasa)
    if not terms or any(t not in pos_index for t in terms):
        return []
    kandidat = set.intersection(*(set(pos_index[t]) for t in terms))
    hasil = []
    for d in sorted(kandidat):
        for p in pos_index[terms[0]][d]:
            if all(p + i in pos_index[t][d] for i, t in enumerate(terms[1:], start=1)):
                hasil.append(d)
                break
    return hasil

for frasa in ["kecerdasan buatan", "machine learning", "kebakaran hutan", "energi surya"]:
    print(f"{frasa!r:22s} -> {phrase_search(frasa, pos_index)}")


'kecerdasan buatan'    -> ['D02', 'D23']
'machine learning'     -> ['D05']
'kebakaran hutan'      -> ['D15', 'D16']
'energi surya'         -> ['D36']


## Jawaban tugas Modul 2

1. Bandingkan `|V| × N` sel matriks penuh terhadap `Σ df(term)` entri posting inverted index. Yang dihitung adalah **entri logis**, bukan ukuran penyimpanan byte; overhead dictionary tidak termasuk.
2. Term dengan `df = 1` bersifat diskriminatif karena hanya muncul pada satu dokumen, tetapi berisiko kurang berguna jika berupa salah ketik, angka acak, atau artefak markup.
3. Menghapus stopword sebelum menghitung posisi dapat menghasilkan pencocokan frasa palsu (kata yang semula terpisah stopword menjadi bersebelahan). Kode berikut membangun positional index dari hasil case folding, tokenisasi, dan stemming **tanpa menghapus stopword**, serta memperlihatkan perbedaannya. Frasa `sistem temu kembali informasi` dan `gerakan 3M` belum tentu ada dalam 50 dokumen; cek dengan data aktual, jangan mengarang hasil temuan.
4. `proximity_search(t1,t2,k)` mencari dokumen yang memuat kedua term dalam jarak posisi maksimum `k` (selisih indeks posisi absolut, urutan bebas).


In [ ]:
# TUGAS 1 — penghematan jumlah entri
full_entries = len(vocab) * len(docs)
posting_entries = sum(len(postings) for postings in index.values())
ratio = full_entries / posting_entries if posting_entries else 0
print(f"Matriks penuh: {len(vocab)} × {len(docs)} = {full_entries} sel")
print(f"Inverted index: {posting_entries} entri posting")
print(f"Perbandingan: {ratio:.2f}× lebih sedikit entri (di luar overhead kamus)")

# TUGAS 2 — tampilkan 10 term dengan document frequency = 1
hapax_df = [(term, next(iter(postings))) for term, postings in index.items() if len(postings) == 1]
print(f"\nTerm df=1: {len(hapax_df)}; contoh 10 term:")
display(pd.DataFrame(hapax_df[:10], columns=["term", "satu-satunya dokumen"]))

# TUGAS 3 — posisi dipertahankan dengan stopwords masih ada
# Sastrawi stemming sama seperti modul 1, hanya stopword removal dimatikan.
docs_pos_lengkap = {d: preprocess(t, use_stopword=False, use_stemming=True)
                    for d, t in teks.items()}
pos_index_lengkap = build_positional_index(docs_pos_lengkap)
phrases = ["sistem temu kembali informasi", "gerakan 3M", "kecerdasan buatan", "harga properti"]
print("\nPhrase search dengan/tanpa stopword removal:")
for frasa in phrases:
    tanpa = phrase_search(frasa, pos_index)
    lengkap = phrase_search(frasa, pos_index_lengkap,
                            tokenizer=lambda t: preprocess(t, use_stopword=False, use_stemming=True))
    print(f"{frasa!r}: tanpa SW={tanpa}, posisi asli={lengkap}")

# Uji buatan yang secara eksplisit menunjukkan masalah posisi yang bergeser
uji_frasa = {"TEST": "sistem untuk temu kembali informasi"}
uji_drop = build_positional_index({d: preprocess(t) for d,t in uji_frasa.items()})
uji_full = build_positional_index({d: preprocess(t,use_stopword=False) for d,t in uji_frasa.items()})
print("\nTes stopword di antara kata:")
print("Setelah SW dihapus:", phrase_search("sistem temu kembali informasi", uji_drop))
print("Sebelum SW dihapus:", phrase_search("sistem temu kembali informasi", uji_full,
       tokenizer=lambda t: preprocess(t,use_stopword=False)))

# TUGAS 4 — proximity dengan index posisi asli (stopword tidak dibuang)
def proximity_search(t1, t2, k, positional_index=pos_index_lengkap):
    if k < 0:
        raise ValueError("k harus >= 0")
    a = preprocess(t1, use_stopword=False)
    b = preprocess(t2, use_stopword=False)
    if len(a) != 1 or len(b) != 1:
        raise ValueError("t1 dan t2 harus masing-masing satu term")
    p1, p2 = positional_index.get(a[0], {}), positional_index.get(b[0], {})
    hasil = []
    for doc_id in sorted(set(p1) & set(p2)):
        if any(abs(x-y) <= k and (a[0]!=b[0] or x!=y)
               for x in p1[doc_id] for y in p2[doc_id]):
            hasil.append(doc_id)
    return hasil

for t1,t2,k in [("kecerdasan", "buatan", 1), ("kebakaran", "hutan", 2), ("energi", "surya", 2)]:
    print(f"proximity_search({t1!r}, {t2!r}, {k}): {proximity_search(t1,t2,k)}")
assert "D02" in proximity_search("kecerdasan", "buatan", 1)
